In [1]:
import sys
from pathlib import Path
import warnings
from sklearn.exceptions import ConvergenceWarning


ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)


from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.codi.models import CODI


warnings.filterwarnings("ignore", category=ConvergenceWarning)          # sklearn MLP
warnings.filterwarnings("ignore", message="Parameters: {")              # XGBoost unused params
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")


dataset_name = "car"   

dataset_path = ROOT / "raw_data" / f"{dataset_name}.csv"
output_path = ROOT / "discretized_data" / f"{dataset_name}.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Preprocess (same style you used for nursery)
print(f"Discretizing {dataset_path} -> {output_path}")
discretize_preprocess(str(dataset_path), str(output_path))

# Paths for pipeline
input_csv     = str(output_path)
output_dir    = str(ROOT / "sample_data" / dataset_name)
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / dataset_name / "codi")

print("input_csv    :", input_csv)
print("output_dir   :", output_dir)
print("real_test_dir:", real_test_dir)
print("synthetic_dir:", synthetic_dir)

class CoDiCar(CODI):
    def __init__(self):
        super().__init__(
            # Diffusion hyperparameters
            n_steps=50,        # you can increase (e.g. 100) if you want stronger sampling
            beta_1=1e-5,
            beta_T=0.02,

            # Network architecture
            encoder_dim_con=(64, 128, 256),
            encoder_dim_dis=(64, 128, 256),
            nf_con=16,
            nf_dis=64,
            activation="relu",

            # Training hyperparameters
            epochs=30,         # bump up (e.g. 50) if you want more training for car (1781 rows)
            batch_size=512,
            lr_con=2e-3,
            lr_dis=2e-3,
            grad_clip=1.0,

            # Contrastive learning weights
            lambda_con=0.2,
            lambda_dis=0.2,

            # Misc
            random_state=42,
            device=None,       # auto: cuda if available, otherwise cpu
        )

pipeline = TrainTestSplitPipeline(
    model=lambda: CoDiCar()
)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print("\nPipeline result (should include TSTR metrics + result path):")
print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic


INFO:katabatic.models.codi.models:================================================================================


Discretizing C:\Users\Prabu\Downloads\Katabatic\raw_data\car.csv -> C:\Users\Prabu\Downloads\Katabatic\discretized_data\car.csv
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\car.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\car.csv
input_csv    : C:\Users\Prabu\Downloads\Katabatic\discretized_data\car.csv
output_dir   : C:\Users\Prabu\Downloads\Katabatic\sample_data\car
real_test_dir: C:\Users\Prabu\Downloads\Katabatic\sample_data\car
synthetic_dir: C:\Users\Prabu\Downloads\Katabatic\synthetic\car\codi
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)


INFO:katabatic.models.codi.models:Training CoDi Model
INFO:katabatic.models.codi.models:================================================================================
INFO:katabatic.models.codi.models:Loaded training data: (1382, 7)
INFO:katabatic.models.codi.models:Schema: 0 continuous, 7 categorical columns
INFO:katabatic.models.codi.models:Building models: con_dim=1, cat_dim=25
INFO:katabatic.models.codi.models:Continuous model params: 166,427
INFO:katabatic.models.codi.models:Discrete model params: 358,001
INFO:katabatic.models.codi.models:
Training for 30 epochs...
INFO:katabatic.models.codi.models:Epoch 1/30: loss_con=0.0000, loss_dis=45.5304
INFO:katabatic.models.codi.models:Epoch 5/30: loss_con=0.0000, loss_dis=44.5342
INFO:katabatic.models.codi.models:Epoch 10/30: loss_con=0.0000, loss_dis=44.3873
INFO:katabatic.models.codi.models:Epoch 15/30: loss_con=0.0000, loss_dis=44.2641
INFO:katabatic.models.codi.models:Epoch 20/30: loss_con=0.0000, loss_dis=44.0832
INFO:katabatic.mod


Results saved to: Results\car\codi_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7081
F1 Score: 0.6288

MLP:
Accuracy: 0.6994
F1 Score: 0.6277

RF:
Accuracy: 0.7110
F1 Score: 0.6115

XGBoost:
Accuracy: 0.7168
F1 Score: 0.6191

Pipeline result (should include TSTR metrics + result path):
Train test split pipeline executed successfully.


In [2]:
import shutil

src = r"C:\Users\Prabu\Downloads\Katabatic\Results\car\codi_tstr.csv"
dst = r"C:\Users\Prabu\Downloads\codi_tstr.csv"

shutil.copy(src, dst)
print("Copied to:", dst)

Copied to: C:\Users\Prabu\Downloads\codi_tstr.csv
